In [1]:
import os
import shutil
import sys

print("--- Installing required packages ---")
!pip install --no-cache-dir -q peft rasterio faiss-gpu huggingface_hub wandb 2> /dev/null

src_path = "/kaggle/input/datasets/suryansh10100/copfmr123"
dest_path = "/kaggle/working/copfm_retrieval"

if os.path.exists(dest_path):
    shutil.rmtree(dest_path)
shutil.copytree(src_path, dest_path)
print("✅ Codebase copied successfully.")

os.chdir(dest_path)
sys.path.insert(0, dest_path)
sys.path.insert(0, os.path.join(dest_path, "copernicus_fm_src"))

print("--- Fetching pre-trained weights ---")
!python setup.py
print("✅ CopFM Weights ready.")

--- Installing required packages ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 143.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 217.1 MB/s eta 0:00:00
✅ Codebase copied successfully.
--- Fetching pre-trained weights ---
🚀 Starting Automated Setup for CR-JEPA Foundation Model...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
CopernicusFM_ViT_base_varlang_e100.pth: 100%|█| 558M/558M [00:05<00:00, 104MB/s]
✅ Weights successfully downloaded!
✅ Setup Complete! You are ready to train.
✅ CopFM Weights ready.


In [2]:
import os
import yaml

os.chdir("/kaggle/working/copfm_retrieval")

# 1. Hotfix DataParallel Bug
loss_file = "src/losses/__init__.py"
with open(loss_file, "r") as f: content = f.read()
old_logic = "l_sigreg = total_sigreg_loss(model_output, _sigreg_fn)"
new_logic = """l_sigreg = total_sigreg_loss(model_output, _sigreg_fn)\n    if l_sigreg.dim() > 0:\n        l_sigreg = l_sigreg.mean()"""
if "l_sigreg.dim() > 0" not in content:
    content = content.replace(old_logic, new_logic)
    with open(loss_file, "w") as f: f.write(content)

# 2. Config & 12-CHANNEL BUG FIX
config_path = 'configs/ben14k.yaml'
with open(config_path, 'r') as f: config = yaml.safe_load(f)

config['data']['modality_b'] = 'S2' # THE CRITICAL FIX: Match 12 channels for S2!
config['data']['root_dir'] = '/kaggle/input/datasets/narendraaironi/bigearthnet-14k/BEN_14k'
config['data']['batch_size'] = 96
config['training']['epochs'] = 150

with open("configs/ben14k_final.yaml", "w") as f: yaml.dump(config, f)

# 3. Setup LoRA
!sed -i 's/lora_r: int = 16/lora_r: int = 64/g' src/models/backbone.py
!sed -i 's/lora_alpha: int = 32/lora_alpha: int = 128/g' src/models/backbone.py

print("✅ Setup complete! Channel bug fixed, LoRA set to 64, Batch set to 96.")

✅ Setup complete! Channel bug fixed, LoRA set to 64, Batch set to 96.


In [3]:
import os

os.chdir("/kaggle/working/copfm_retrieval")
os.environ["WANDB_MODE"] = "disabled"

print("🔥 Starting FRESH TRAINING from Scratch (Rank=64, Batch=96, Epochs=150)...")
!python train.py --config configs/ben14k_final.yaml

🔥 Starting FRESH TRAINING from Scratch (Rank=64, Batch=96, Epochs=150)...
Loading metadata from /kaggle/input/datasets/narendraaironi/bigearthnet-14k/BEN_14k/metadata.parquet...
BEN14KDataset (split=train) initialized with 7180 real samples.
Loading metadata from /kaggle/input/datasets/narendraaironi/bigearthnet-14k/BEN_14k/metadata.parquet...
BEN14KDataset (split=validation) initialized with 3255 real samples.
CopFMBackbone initialized. Mode: lora
Total Params: 141819530 | Trainable Params: 2359296
Using 2 GPUs for training!
Starting training loop...
Epoch 1/150:   0%|                                       | 0/74 [00:10<?, ?it/s]
Traceback (most recent call last):
  File "/kaggle/working/copfm_retrieval/train.py", line 214, in <module>
    main(args.config)
  File "/kaggle/working/copfm_retrieval/train.py", line 146, in main
    output = model(
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return s

In [4]:
%%writefile /kaggle/working/copfm_retrieval/eval_comprehensive.py
import os, time, torch, yaml, faiss
import numpy as np
from tqdm import tqdm
from src.datasets.ben14k import BEN14KDataset
from src.models.copfm_retrieval import CopFMRetrieval
from src.wavelengths import get_wavelengths

def evaluate_comprehensive(query_embs, query_labels, gallery_embs, gallery_labels, ks=[5, 10], is_same_modal=False):
    D = query_embs.shape[1]
    index = faiss.IndexFlatIP(D)
    index.add(gallery_embs.astype(np.float32))
    
    search_k = max(ks) + 1 if is_same_modal else max(ks)
    start_time = time.time()
    distances, indices = index.search(query_embs.astype(np.float32), search_k)
    avg_time_ms = ((time.time() - start_time) / len(query_embs)) * 1000.0
    
    results = {}
    for k in ks:
        all_f1s = []
        for i in range(len(query_embs)):
            q_label = query_labels[i]
            q_sum = q_label.sum()
            valid_retrievals = 0
            query_f1s = []
            
            for j in range(search_k):
                r_idx = indices[i, j]
                if is_same_modal and r_idx == i: continue
                if valid_retrievals >= k: break
                
                r_label = gallery_labels[r_idx]
                r_sum = r_label.sum()
                intersection = np.dot(q_label, r_label)
                if r_sum == 0 or q_sum == 0: f1_qr = 0.0
                else:
                    p_qr = intersection / r_sum
                    r_qr = intersection / q_sum
                    f1_qr = 0.0 if (p_qr + r_qr) == 0 else 2 * p_qr * r_qr / (p_qr + r_qr + 1e-8)
                query_f1s.append(f1_qr)
                valid_retrievals += 1
            all_f1s.append(np.mean(query_f1s))
        results[f'f1@{k}'] = np.mean(all_f1s) * 100.0
    return results, avg_time_ms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with open('configs/ben14k_final.yaml', 'r') as f: config = yaml.safe_load(f)

# Wait for 150 epochs to finish before evaluating!
ckpt_path = 'checkpoints/epoch_150.pth'

model = CopFMRetrieval(config['model'])
if torch.cuda.device_count() > 1: model = torch.nn.DataParallel(model)
model.to(device)

print(f"Loading weights from {ckpt_path}...")
checkpoint = torch.load(ckpt_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

test_loader = torch.utils.data.DataLoader(BEN14KDataset(config['data']['root_dir'], split='test'), batch_size=128, shuffle=False, num_workers=4)
wl_a, bw_a = get_wavelengths('S1')
wl_b, bw_b = get_wavelengths('S2')

s1_cross, s2_cross, s1_uni, s2_uni, all_labels = [], [], [], [], []
print("Extracting features for the TEST set...")
with torch.no_grad():
    for batch in tqdm(test_loader):
        img_a, img_b = batch['s1'].to(device), batch['s2'].to(device)
        B, _, H, W = img_a.shape
        mask = torch.zeros((B, (H//16)*(W//16)), dtype=torch.bool, device=device)
        out = model(img_a, img_b, wl_a, wl_b, bw_a, bw_b, mask, mask)
        
        e_c_a, e_c_b = out['e_cross_a'].cpu().numpy(), out['e_cross_b'].cpu().numpy()
        s1_cross.append(e_c_a / np.linalg.norm(e_c_a, axis=1, keepdims=True))
        s2_cross.append(e_c_b / np.linalg.norm(e_c_b, axis=1, keepdims=True))
        
        e_u_a, e_u_b = out['e_uni_a'].cpu().numpy(), out['e_uni_b'].cpu().numpy()
        s1_uni.append(e_u_a / np.linalg.norm(e_u_a, axis=1, keepdims=True))
        s2_uni.append(e_u_b / np.linalg.norm(e_u_b, axis=1, keepdims=True))
        all_labels.append(batch['label'].numpy())

s1_cross, s2_cross = np.concatenate(s1_cross, axis=0), np.concatenate(s2_cross, axis=0)
s1_uni, s2_uni = np.concatenate(s1_uni, axis=0), np.concatenate(s2_uni, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

print("\n" + "="*50 + "\n🏆 CROSS-MODAL EVALUATION 🏆\n" + "="*50)
res, t = evaluate_comprehensive(s1_cross, all_labels, s2_cross, all_labels, is_same_modal=False)
print(f"S1 → S2 (SAR to Opt) | F1@5: {res['f1@5']:.2f}% | F1@10: {res['f1@10']:.2f}% | Time/Query: {t:.4f} ms")
res, t = evaluate_comprehensive(s2_cross, all_labels, s1_cross, all_labels, is_same_modal=False)
print(f"S2 → S1 (Opt to SAR) | F1@5: {res['f1@5']:.2f}% | F1@10: {res['f1@10']:.2f}% | Time/Query: {t:.4f} ms")

print("\n" + "="*50 + "\n🏅 SAME-MODAL EVALUATION (Self-Matches Filtered) 🏅\n" + "="*50)
res, t = evaluate_comprehensive(s1_uni, all_labels, s1_uni, all_labels, is_same_modal=True)
print(f"S1 → S1 (SAR to SAR) | F1@5: {res['f1@5']:.2f}% | F1@10: {res['f1@10']:.2f}% | Time/Query: {t:.4f} ms")
res, t = evaluate_comprehensive(s2_uni, all_labels, s2_uni, all_labels, is_same_modal=True)
print(f"S2 → S2 (Opt to Opt) | F1@5: {res['f1@5']:.2f}% | F1@10: {res['f1@10']:.2f}% | Time/Query: {t:.4f} ms")

Writing /kaggle/working/copfm_retrieval/eval_comprehensive.py


In [5]:
import os

os.chdir("/kaggle/working/copfm_retrieval")
!python eval_comprehensive.py

CopFMBackbone initialized. Mode: lora
Total Params: 141819530 | Trainable Params: 2359296
Loading weights from checkpoints/epoch_150.pth...
Traceback (most recent call last):
  File "/kaggle/working/copfm_retrieval/eval_comprehensive.py", line 57, in <module>
    checkpoint = torch.load(ckpt_path, map_location=device)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1500, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 768, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 749, in __init__
    super().__init__(open(name, mode))  # noqa: SIM115
                     ^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'checkpoin